# 04_model_training.ipynb

Train Random Forest and XGBoost classifiers on the preprocessed heart disease dataset, evaluate both models, save the best model, and display feature importance.

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

# XGBoost import with fallback explanation if not installed
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError('xgboost is not installed. Install it with `pip install xgboost`.') from exc

# Paths
clean_path = os.path.join('data', 'processed', 'cleaned_data.csv')
scaler_path = os.path.join('models', 'scaler.pkl')
rf_path = os.path.join('models', 'random_forest.pkl')
xgb_path = os.path.join('models', 'xgboost.pkl')
best_path = os.path.join('models', 'best_model.pkl')
os.makedirs('models', exist_ok=True)

# Load the cleaned dataset
if not os.path.exists(clean_path):
    raise FileNotFoundError(f'Cleaned dataset not found at {clean_path}. Please run preprocessing first.')
df = pd.read_csv(clean_path)
print('Loaded cleaned dataset with shape:', df.shape)

In [ ]:
# Features and target selection
feature_columns = [
    'age_years', 'gender', 'height', 'weight', 'BMI',
    'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active'
]
target_column = 'cardio'

# Verify that all required columns exist
missing_cols = [col for col in feature_columns + [target_column] if col not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

X = df[feature_columns].copy()
y = df[target_column].copy()

print('Selected features:', feature_columns)
print('Target:', target_column)
print('X shape:', X.shape)
print('y shape:', y.shape)

## Why these steps matter
- **Random Forest:** builds many decision trees on random subsets of data and averages their predictions.
- **XGBoost:** builds trees sequentially, correcting prior errors and using gradient boosting to improve performance.
- **Model training:** `fit()` learns patterns from `X_train` and `y_train`.
- **Prediction:** `predict()` applies the learned model to new data (`X_test`).
- **Evaluation metrics:** measure how well each model performs on unseen data.

In [ ]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

# Ensure the scaler exists and can be loaded from previous step
if not os.path.exists(scaler_path):
    raise FileNotFoundError(f'Scaler not found at {scaler_path}. Run the feature selection notebook first.')
scaler = joblib.load(scaler_path)
print('Loaded scaler from', scaler_path)

# Transform features with StandardScaler
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape:', X_test_scaled.shape)

In [ ]:
# Create a helper function to evaluate models
def evaluate_model(name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)
    summary = {
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
    }
    print(f'\n{name} Evaluation:')
    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score:', f1)
    print('Confusion matrix:\n', cm)
    print('Classification report:\n', classification_report(y_true, y_pred))
    return summary

results = []
best_model = None
best_name = None
best_score = -1


In [ ]:
# Train Random Forest classifier
rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train_scaled, y_train)  # fit() is called
rf_pred = rf.predict(X_test_scaled)  # predict() is called
joblib.dump(rf, rf_path)
print('Saved Random Forest model to', rf_path)
rf_summary = evaluate_model('Random Forest', y_test, rf_pred)
results.append(rf_summary)
rf_importance = rf.feature_importances_

In [ ]:
# Train XGBoost classifier
xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
xgb.fit(X_train_scaled, y_train)  # fit() is called
xgb_pred = xgb.predict(X_test_scaled)  # predict() is called
joblib.dump(xgb, xgb_path)
print('Saved XGBoost model to', xgb_path)
xgb_summary = evaluate_model('XGBoost', y_test, xgb_pred)
results.append(xgb_summary)
xgb_importance = xgb.feature_importances_

In [ ]:
# Compare model performance and save the best model
comparison = pd.DataFrame(results)
print('Comparison table:')
print(comparison.to_string(index=False))

# Select the best model by F1 Score first, then Accuracy
best_index = comparison.sort_values(['F1 Score', 'Accuracy'], ascending=False).index[0]
best_name = comparison.loc[best_index, 'Model']
if best_name == 'Random Forest':
    best_model = rf
    best_importances = rf_importance
else:
    best_model = xgb
    best_importances = xgb_importance

joblib.dump(best_model, best_path)
print(f'Saved best model ({best_name}) to {best_path}')

importance_df = pd.DataFrame({'feature': feature_columns, 'importance': best_importances}).sort_values('importance', ascending=False)
print('\nTop 5 important features for the best model:')
print(importance_df.head(5).to_string(index=False))

print('\nBest model name:', best_name)
print('Best accuracy:', comparison.loc[best_index, 'Accuracy'])


## Model explanations
- **Random Forest:** an ensemble of decision trees that reduces overfitting by averaging many tree predictions.
- **XGBoost:** a boosting method that builds trees sequentially, where each new tree corrects errors made by previous trees.
- **Advantages of Random Forest:** robust to noise, less tuning required, works well on tabular data.
- **Disadvantages of Random Forest:** can be slower for very large datasets, less interpretable than a single tree.
- **Advantages of XGBoost:** often more accurate, handles missing values, strong regularization.
- **Disadvantages of XGBoost:** more parameters to tune, can overfit if not regularized properly.